# LINEUP - Quickstart

Evaluate any per-passage attribution method against LINEUP's non-circular causal ground truth in a few lines. This runs on the bundled 6-case sample (no GPU, no download); point it at a real run's `outputs/` or the HuggingFace dataset for full results.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src' / 'lineup').exists() and (ROOT.parent / 'src' / 'lineup').exists():
    ROOT = ROOT.parent          # allow running from notebooks/
sys.path.insert(0, str(ROOT / 'src'))
SAMPLE = ROOT / 'app' / 'sample'
print('repo root:', ROOT)

## 1. Score the built-in methods

Load the ground-truth roles and the bundled predictions, then `evaluate()` -- the same scoring used for the paper, so your numbers are comparable to the leaderboard.

In [ ]:
from lineup.data.serialization import read_roles, read_predictions
from lineup.evaluate import evaluate, leaderboard_markdown, no_culprit_rate

cases = [c for c in read_roles(SAMPLE / 'roles.jsonl') if not c.original_correct]
preds = read_predictions(SAMPLE / 'predictions.jsonl')
print(leaderboard_markdown(evaluate(cases, preds), no_culprit=no_culprit_rate(cases)))

## 2. Plug in your own method

Your method sees the question and the passages and returns a score per passage (higher = more responsible). Here is a trivial example -- score each passage by word overlap with the model's (wrong) answer -- wired through the same `evaluate()`.

In [ ]:
import re
from lineup.data.serialization import read_scenarios, read_generations
from lineup.evaluate import predictions_from_scores

scen = {s.qid: s for s in read_scenarios(SAMPLE / 'scenarios.jsonl')}
gen = {g.qid: g for g in read_generations(SAMPLE / 'generations.jsonl')}

def my_method(question, passages, answer):
    atok = set(re.findall(r'\w+', answer.lower()))
    return {cid: len(atok & set(re.findall(r'\w+', text.lower()))) for cid, text in passages}

scores_by_qid = {}
for case in cases:
    passages = [(c.chunk_id, c.text) for c in scen[case.qid].chunks]
    scores_by_qid[case.qid] = my_method(case.question, passages, gen[case.qid].model_answer)

my_preds = predictions_from_scores('my_overlap_method', scores_by_qid)
print(leaderboard_markdown(evaluate(cases, my_preds)))

## Next

- Point `SAMPLE` at a real run's `outputs/` (or `load_dataset('<org>/lineup')`) for full results.
- The headline LINEUP surfaces: a large `recall@k - recall@1` gap and a low single-culprit AUROC mean your method should return a **set** (with abstention / conformal calibration), not one passage.
- See `docs/evaluate.md` for metric definitions and the published leaderboard.